In [ ]:
# Cell 1: Secrets
import os
from google.colab import userdata






In [2]:
# Cell 2: Setup
!git clone https://github.com/sk3feel/hidden-data-reproduction-multimodal.git /content/course_work2026 2>/dev/null || (cd /content/course_work2026 && git pull)
%cd /content/course_work2026
!pip install -q -r requirements.txt
!pip install -q transformers==4.49.0 accelerate Pillow pandas tqdm einops timm comet_ml huggingface_hub
!python src/download_from_hf.py --repo-id sk3feel/docvqa-privacy-data

from pathlib import Path

assert Path("artifacts/finetuning_generative/train_florence2.jsonl").exists()
assert Path("artifacts/docqa_recovery/benchmark_train/images/original").exists()


/content/course_work2026
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 120.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 106.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-kernel-gateway 2.5.2 requires notebook<7.0,>=5.7.6, but you have notebook 7.5.5 which is incompatible.
artifacts_bundle.tar.gz: 100% 3.04G/3.04

In [3]:
# Cell 3: Imports and constants
import json
import os
import re
from pathlib import Path

import comet_ml
import pandas as pd
import torch
from PIL import Image, ImageFile
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoProcessor
from huggingface_hub import HfApi, login
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

MODEL_NAME = "microsoft/Florence-2-base"
EPOCHS = 30
BATCH_SIZE = 2
LR = 1e-5
CHECKPOINT_EPOCHS = [1, 5, 15, 30]
HF_MODEL_REPO = "sk3feel/docvqa-privacy-checkpoints"
TRAIN_JSONL = Path("artifacts/finetuning_generative/train_florence2.jsonl")
VALIDATION_JSONL = Path("artifacts/finetuning_generative/validation_florence2.jsonl")

experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
experiment.set_name("florence2-finetune")
experiment.log_parameters({"model": MODEL_NAME, "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE})
login(token=os.environ["HF_TOKEN"])
api = HfApi()
api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=True, exist_ok=True)


COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/scfeel/qwen3-1/10300a312a1d4bcb9f3983d47dbd7d13

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


RepoUrl('https://huggingface.co/sk3feel/docvqa-privacy-checkpoints', endpoint='https://huggingface.co', repo_type='model', repo_id='sk3feel/docvqa-privacy-checkpoints')

In [4]:
# Cell 4: Load data
QUESTION_RE = re.compile(r"<Question>(.*?)</Question>", flags=re.IGNORECASE | re.DOTALL)


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def extract_question(row: dict) -> str:
    if row.get("question"):
        return str(row["question"]).strip()
    if row.get("prompt"):
        return str(row["prompt"]).strip()
    task_prompt = str(row.get("task_prompt", "")).strip()
    match = QUESTION_RE.search(task_prompt)
    if match:
        return match.group(1).strip()
    return task_prompt.replace("<VQA>", "").strip()


def resolve_image_path(raw_path: str, split: str) -> Path:
    path = Path(raw_path)
    if path.exists():
        return path
    image_name = raw_path.replace("\\", "/").split("/")[-1]
    split_dir = "benchmark_train" if split == "train" else "benchmark"
    candidate = Path("artifacts") / "docqa_recovery" / split_dir / "images" / "original" / image_name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f"Image not found: {raw_path}")


def prepare_records(rows: list[dict], default_split: str) -> list[dict]:
    records = []
    for row in rows:
        split = str(row.get("split", default_split))
        records.append(
            {
                "image_path": str(resolve_image_path(str(row["image_path"]), split)),
                "question": extract_question(row),
                "answer": str(row["answer"]).strip(),
                "split": split,
            }
        )
    return records


train_records = prepare_records(load_jsonl(TRAIN_JSONL), "train")
validation_records = prepare_records(load_jsonl(VALIDATION_JSONL), "validation")

print({"train_records": len(train_records), "validation_records": len(validation_records)})

sample_rows = []
for rec in train_records[:3]:
    with Image.open(rec["image_path"]).convert("RGB") as image:
        sample_rows.append(
            {
                "question": rec["question"],
                "answer": rec["answer"],
                "image_path": rec["image_path"],
                "image_size": image.size,
            }
        )

pd.DataFrame(sample_rows)


{'train_records': 800, 'validation_records': 1612}


,question,answer,image_path,image_size
0,What is the type of organization?,Corporation,artifacts/docqa_recovery/benchmark_train/image...,"(1692, 2245)"
1,what is the price at bottom of the page ?,$1.90,artifacts/docqa_recovery/benchmark_train/image...,"(1684, 2186)"
2,Who has MCH as area of special emphasis and fr...,"BANTA, JAMES E.",artifacts/docqa_recovery/benchmark_train/image...,"(1784, 2291)"


In [5]:
# Cell 5: Dataset
class Florence2Dataset(Dataset):
    def __init__(self, records, processor):
        self.records = records
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image = Image.open(rec["image_path"]).convert("RGB")
        prompt = f"<VQA>{rec['question']}"
        inputs = self.processor(text=prompt, images=image, return_tensors="pt", padding=True, truncation=True)
        labels = self.processor.tokenizer(text=rec["answer"], return_tensors="pt", padding=True, truncation=True).input_ids
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "pixel_values": inputs["pixel_values"].squeeze(),
            "labels": labels.squeeze(),
        }


def make_collate_fn(processor):
    pad_token_id = processor.tokenizer.pad_token_id or 0

    def collate_fn(batch):
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [item["input_ids"] for item in batch],
            batch_first=True,
            padding_value=pad_token_id,
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [item["labels"] for item in batch],
            batch_first=True,
            padding_value=-100,
        )
        pixel_values = torch.stack([item["pixel_values"] for item in batch])
        return {
            "input_ids": input_ids,
            "pixel_values": pixel_values,
            "labels": labels,
        }

    return collate_fn


In [6]:
# Cell 6: Training loop
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("GPU runtime is required")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to("cuda")
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
dataset = Florence2Dataset(train_records, processor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=make_collate_fn(processor))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss, valid_batches = 0, 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}"):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        outputs = model(input_ids=batch["input_ids"], pixel_values=batch["pixel_values"], labels=batch["labels"])
        loss = outputs.loss
        if not loss.isfinite():
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        valid_batches += 1
    avg_loss = epoch_loss / max(valid_batches, 1)
    print(f"Epoch {epoch}, loss: {avg_loss:.4f}")
    experiment.log_metric("train_loss", avg_loss, epoch=epoch)

    if epoch in CHECKPOINT_EPOCHS:
        ckpt = f"checkpoints/florence2/epoch_{epoch}"
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt)
        processor.save_pretrained(ckpt)
        api.upload_folder(folder_path=ckpt, repo_id=HF_MODEL_REPO, path_in_repo=f"florence2/epoch_{epoch}", repo_type="model")

        model.eval()
        correct = 0
        for rec in train_records[:20]:
            img = Image.open(rec["image_path"]).convert("RGB")
            inp = processor(text=f"<VQA>{rec['question']}", images=img, return_tensors="pt").to("cuda")
            with torch.no_grad():
                gen = model.generate(**inp, max_new_tokens=50)
            pred = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
            if pred.lower() == rec["answer"].lower():
                correct += 1
        experiment.log_metric("sanity_em", correct / 20, epoch=epoch)
        model.train()


config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


AttributeError: 'Florence2LanguageConfig' object has no attribute 'forced_bos_token_id'

In [ ]:
# Cell 7: Final sanity
final_ckpt = Path("checkpoints/florence2/epoch_30")
assert final_ckpt.exists(), f"Missing checkpoint: {final_ckpt}"

final_model = AutoModelForCausalLM.from_pretrained(final_ckpt, trust_remote_code=True).to("cuda")
final_processor = AutoProcessor.from_pretrained(final_ckpt, trust_remote_code=True)
final_model.eval()


def normalize_text(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


def run_final_sanity(records: list[dict], split_name: str) -> tuple[list[dict], float]:
    rows = []
    correct = 0
    for rec in records[:100]:
        img = Image.open(rec["image_path"]).convert("RGB")
        inp = final_processor(text=f"<VQA>{rec['question']}", images=img, return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen = final_model.generate(**inp, max_new_tokens=50)
        pred = final_processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
        is_match = normalize_text(pred) == normalize_text(rec["answer"])
        correct += int(is_match)
        rows.append(
            {
                "split": split_name,
                "question": rec["question"],
                "gold": rec["answer"],
                "prediction": pred,
                "exact_match": is_match,
            }
        )
    return rows, correct / max(len(rows), 1)


seen_rows, seen_em = run_final_sanity(train_records, "seen")
unseen_rows, unseen_em = run_final_sanity(validation_records, "unseen")

print({"seen_em": seen_em, "unseen_em": unseen_em})

sanity_final_df = pd.DataFrame(seen_rows + unseen_rows)
experiment.log_metric("sanity_final_seen_em", seen_em)
experiment.log_metric("sanity_final_unseen_em", unseen_em)
experiment.log_table("sanity_final.csv", sanity_final_df)
display(sanity_final_df.head(10))


In [ ]:
# Cell 8: Finish
experiment.end()
